# ⚙️ Shared Configuration

**Ovaj notebook sadrži zajedničku konfiguraciju za ceo projekat.**

Svi ostali notebook-i ga uključuju pomoću:
```python
%run ../includes/config
```

Šta se nalazi ovde:
- Katalog i schema imena
- ADLS putanje
- Definicije izvora podataka (SOURCE_CONFIGS)
- Helper funkcije koje se koriste u više notebook-a

In [0]:
# ==============================================================================
# CENTRALNA KONFIGURACIJA PROJEKTA
# ==============================================================================
# Ovaj fajl se importuje u svaki notebook sa: %run ../includes/config
# Sve promenljive definisane ovde su dostupne u notebook-u koji radi %run.
# ==============================================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime

# ==============================================================================
# KATALOG I SCHEMA
# ==============================================================================
# hive_metastore — koristi Databricks-managed storage (DBFS)
# used_vehicle_databricks_ws — Unity Catalog (vidljiv u Catalog Explorer)

CATALOG = "hive_metastore"          # Primarni storage (pouzdano)
SCHEMA_BRONZE = "bronze"             # Sirovi podaci
SCHEMA_SILVER = "silver"             # Očišćeni podaci
SCHEMA_GOLD = "gold"                 # Business-ready podaci

# Unity Catalog (za vidljivost u Catalog Explorer-u)
UC_CATALOG = "used_vehicle_databricks_ws"

# ==============================================================================
# ADLS PUTANJE (Azure Data Lake Storage Gen2)
# ==============================================================================
# NAPOMENA: Ove putanje rade SAMO dok je Azure subscription aktivna.
# Podaci su već sačuvani u Delta tabelama — ove putanje su samo za re-ingestion.

ADLS_BASE = "abfss://raw@usedvehicledl.dfs.core.windows.net"

# ==============================================================================
# IZVOR PODATAKA - Konfiguracija svih source sistema
# ==============================================================================

SOURCE_CONFIGS = [
    {
        "table_name": "first_source_system",
        "path": f"{ADLS_BASE}/source_systems/FirstSourceSystem",
        "format": "csv",
        "options": {"header": "true", "inferSchema": "true"}
    },
    {
        "table_name": "second_source_system",
        "path": f"{ADLS_BASE}/source_systems/SecondSourceSystem",
        "format": "parquet",
        "options": {}
    },
    {
        "table_name": "third_source_system",
        "path": f"{ADLS_BASE}/source_systems/ThirdSourceSystem",
        "format": "avro",
        "options": {}
    },
    {
        "table_name": "fourth_source_system",
        "path": f"{ADLS_BASE}/source_systems/FourthSourceSystem",
        "format": "json",
        "options": {"multiLine": "false"}
    },
    {
        "table_name": "model_to_brand_lookup",
        "path": f"{ADLS_BASE}/lookup/model_to_brand_lookup",
        "format": "csv",
        "options": {"header": "true", "inferSchema": "true"}
    },
    {
        "table_name": "service_book",
        "path": f"{ADLS_BASE}/service_book/ServiceBook",
        "format": "csv",
        "options": {"header": "true", "inferSchema": "true"}
    },
]

In [0]:
# ==============================================================================
# HELPER FUNKCIJE
# ==============================================================================
# Reusable funkcije koje se koriste u više notebook-a.
# ==============================================================================

def get_full_table_name(schema: str, table: str) -> str:
    """Vraća puno ime tabele: catalog.schema.table"""
    return f"{CATALOG}.{schema}.{table}"


def get_bronze_table(table_name: str) -> str:
    """Vraća puno ime bronze tabele."""
    return get_full_table_name(SCHEMA_BRONZE, table_name)


def get_silver_table(table_name: str) -> str:
    """Vraća puno ime silver tabele."""
    return get_full_table_name(SCHEMA_SILVER, table_name)


def get_gold_table(table_name: str) -> str:
    """Vraća puno ime gold tabele."""
    return get_full_table_name(SCHEMA_GOLD, table_name)


def print_section(title: str):
    """Prikazuje formatirani naslov sekcije u outputu."""
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")


def count_nulls(df, columns=None):
    """
    Broji NULL vrednosti za svaku kolonu u DataFrame-u.
    Ako columns=None, proverava sve kolone.
    """
    if columns is None:
        columns = df.columns
    
    null_counts = df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in columns
    ])
    return null_counts


def add_audit_columns(df, source_path: str):
    """
    Dodaje standardne audit kolone svakom DataFrame-u:
    - _ingestion_timestamp: kada su podaci učitani
    - _source_path: odakle su došli
    """
    return df.withColumn("_ingestion_timestamp", F.current_timestamp()) \
             .withColumn("_source_path", F.lit(source_path))


# ==============================================================================
# POTVRDA
# ==============================================================================
print("✅ Config učitan uspešno.")
print(f"   Katalog: {CATALOG}")
print(f"   Bronze:  {CATALOG}.{SCHEMA_BRONZE}")
print(f"   Silver:  {CATALOG}.{SCHEMA_SILVER}")
print(f"   Gold:    {CATALOG}.{SCHEMA_GOLD}")